# D094 — JSON with Python Objects

QuickCart exchanges orders with web applications and other services. JSON is a text format for that exchange; Python's `json` module converts between JSON text and common Python values.

This notebook writes actual JSON files, reads them back, and reconstructs class objects.

## JSON is data, not a Python object dump

JSON supports a small, language-independent type system:

| Python value | JSON value |
|---|---|
| `dict` | object |
| `list`, `tuple` | array |
| `str` | string |
| `int`, `float` | number |
| `True`, `False` | `true`, `false` |
| `None` | `null` |

JSON does not directly represent `set`, `datetime`, `Decimal`, or arbitrary class objects. They must first be converted into supported values.

# 1. Basic encoding and decoding

- `json.dumps(value)` returns JSON text as a Python string.
- `json.loads(text)` parses JSON text into Python values.
- The **s** in `dumps` and `loads` can be remembered as **string**.

In [ ]:
import json


customer = {
    "customer_id": 101,
    "name": "Ananya Rao",
    "active": True,
    "phone": None,
}
product_names = ["Laptop Stand", "Wireless Mouse", "USB-C Cable"]

customer_json = json.dumps(customer, indent=2)
products_json = json.dumps(product_names)

print(customer_json)
print(products_json)

decoded_customer = json.loads(customer_json)
print(type(decoded_customer), decoded_customer["name"])

## Sets are not JSON values

A Python set is unordered and is not part of the JSON specification. Convert it to a list before encoding. Sort first when stable output is useful for testing or version control.

When reading the JSON, explicitly convert the list back to a set if the application requires set behaviour.

In [ ]:
delivery_zones = {"South", "North", "West"}

try:
    json.dumps(delivery_zones)
except TypeError as error:
    print(type(error).__name__, "-", error)

zones_json = json.dumps(sorted(delivery_zones))
restored_zones = set(json.loads(zones_json))

print(zones_json)
print(restored_zones == delivery_zones)

# 2. Writing and reading JSON files

- `json.dump(value, file)` writes JSON to an open text file.
- `json.load(file)` reads JSON from an open text file.
- Open JSON files with `encoding="utf-8"`.
- Use a context manager so files close automatically.
- `indent=2` produces readable files.
- `ensure_ascii=False` writes Unicode characters directly instead of `\u` escape sequences.

In [ ]:
from pathlib import Path


OUTPUT_DIR = Path("output") / "json_examples"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CUSTOMER_PATH = OUTPUT_DIR / "customer.json"

customer = {
    "customer_id": 102,
    "name": "கவிதா",  # Tamil text
    "city": "Bengaluru",
    "preferred_categories": ["Books", "Electronics"],
}

with CUSTOMER_PATH.open("w", encoding="utf-8") as file:
    json.dump(customer, file, indent=2, ensure_ascii=False)

with CUSTOMER_PATH.open("r", encoding="utf-8") as file:
    loaded_customer = json.load(file)

print(CUSTOMER_PATH.resolve())
print(loaded_customer)
print(customer == loaded_customer)

## File overwrite and append behaviour

Opening a file with mode `"w"` replaces its previous content. `json.dump()` does not merge with or update an existing document automatically.

Do not repeatedly append complete JSON values with mode `"a"`; the result usually contains adjacent JSON documents and is not one valid JSON document.

To update a normal JSON file:

1. load the existing value
2. modify the Python value
3. write the complete value back

For append-oriented event data, use a deliberate format such as JSON Lines (`.jsonl`), where each line is a complete JSON object.

In [ ]:
CART_PATH = OUTPUT_DIR / "cart.json"

cart = {"customer_id": 102, "items": ["P101"]}
with CART_PATH.open("w", encoding="utf-8") as file:
    json.dump(cart, file, indent=2)

# Read, modify, and overwrite the complete JSON document.
with CART_PATH.open("r", encoding="utf-8") as file:
    cart = json.load(file)

cart["items"].append("P205")

with CART_PATH.open("w", encoding="utf-8") as file:
    json.dump(cart, file, indent=2)

print(CART_PATH.read_text(encoding="utf-8"))

# 3. JSON with classes and objects

`json.dumps(order)` fails because the encoder does not know which object attributes belong in the public JSON representation.

A class should define an explicit conversion contract:

- `to_dict()` converts an object into JSON-compatible application data.
- `from_dict()` reconstructs an object from decoded data.

Avoid blindly serializing `obj.__dict__`. It may expose internal attributes and tightly couples the JSON format to the class's storage implementation.

In [ ]:
from datetime import datetime, timezone
from decimal import Decimal


class Order:
    def __init__(self, order_id, customer_name, items, total, created_at):
        self.order_id = int(order_id)
        self.customer_name = customer_name
        self.items = list(items)
        self.total = Decimal(str(total))
        self.created_at = created_at

    def __repr__(self):
        return (
            f"Order(order_id={self.order_id!r}, "
            f"customer_name={self.customer_name!r}, items={self.items!r}, "
            f"total={str(self.total)!r}, created_at={self.created_at!r})"
        )

    def to_dict(self):
        return {
            "order_id": self.order_id,
            "customer_name": self.customer_name,
            "items": self.items,
            "total": str(self.total),
            "created_at": self.created_at.isoformat(),
        }

    @classmethod
    def from_dict(cls, data):
        return cls(
            order_id=data["order_id"],
            customer_name=data["customer_name"],
            items=data["items"],
            total=data["total"],
            created_at=datetime.fromisoformat(data["created_at"]),
        )

## Dates and monetary values

JSON has no date type. Store datetimes as ISO 8601 strings and parse them explicitly while rebuilding the object. Prefer timezone-aware datetimes; the `+00:00` suffix below records UTC.

The standard `json` module does not encode `Decimal` directly. Converting a monetary value to a string preserves its exact decimal digits. Converting it to `float` may lose precision.

In [ ]:
order = Order(
    order_id=5001,
    customer_name="José Kumar",
    items=["P101", "P205"],
    total=Decimal("4598.90"),
    created_at=datetime(2026, 7, 28, 10, 30, tzinfo=timezone.utc),
)

try:
    json.dumps(order)
except TypeError as error:
    print(type(error).__name__, "-", error)

order_json = json.dumps(order.to_dict(), indent=2, ensure_ascii=False)
print(order_json)

## Write an object and reconstruct it from the file

Reading JSON produces dictionaries and lists—not an `Order` automatically. Pass the decoded dictionary to `Order.from_dict()` to restore the class object and its `Decimal` and `datetime` values.

In [ ]:
ORDER_PATH = OUTPUT_DIR / "order.json"

with ORDER_PATH.open("w", encoding="utf-8") as file:
    json.dump(order.to_dict(), file, indent=2, ensure_ascii=False)

with ORDER_PATH.open("r", encoding="utf-8") as file:
    order_data = json.load(file)

restored_order = Order.from_dict(order_data)

print(repr(restored_order))
print(type(restored_order))
print(type(restored_order.total))
print(type(restored_order.created_at))
print(restored_order.to_dict() == order.to_dict())

# 4. Custom encoding with `default`

The `default` parameter receives values that the normal encoder cannot serialize. It must return a supported JSON value or raise `TypeError` for an unknown type.

A plain `default` function is generally simpler than subclassing `JSONEncoder`.

In [ ]:
def ecommerce_json_default(value):
    if isinstance(value, Order):
        return {"__type__": "Order", **value.to_dict()}
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, Decimal):
        return str(value)
    if isinstance(value, set):
        return sorted(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


custom_json = json.dumps(
    {"order": order, "tags": {"priority", "paid"}},
    default=ecommerce_json_default,
    indent=2,
    ensure_ascii=False,
)

print(custom_json)

## Automatic reconstruction with `object_hook`

`object_hook` is called for every JSON object decoded into a dictionary. A type marker can tell the hook which dictionaries represent class objects.

Only honor expected type markers from trusted formats. JSON parsing is not the same as validating external input.

In [ ]:
def ecommerce_object_hook(data):
    object_type = data.pop("__type__", None)
    if object_type == "Order":
        return Order.from_dict(data)
    return data


decoded_payload = json.loads(custom_json, object_hook=ecommerce_object_hook)
decoded_order = decoded_payload["order"]

print(repr(decoded_order))
print(type(decoded_order))
print(decoded_order.to_dict() == order.to_dict())

## Overriding `JSONEncoder.default`

`json.JSONEncoder` can be subclassed and its `default()` method overridden. This is method overriding, not operator overloading. Pass the encoder class with `cls=...`.

Use this approach when a reusable encoder class is valuable. For a small conversion policy, the `default=` function shown above is usually clearer.

In [ ]:
class EcommerceJSONEncoder(json.JSONEncoder):
    def default(self, value):
        if isinstance(value, Order):
            return {"__type__": "Order", **value.to_dict()}
        if isinstance(value, (datetime, Decimal)):
            return str(value) if isinstance(value, Decimal) else value.isoformat()
        if isinstance(value, set):
            return sorted(value)
        return super().default(value)


encoded_with_class = json.dumps(
    order,
    cls=EcommerceJSONEncoder,
    indent=2,
    ensure_ascii=False,
)
print(encoded_with_class)

# 5. `json` and `simplejson`

`json` is included with Python and is the correct default for most applications. `simplejson` is a third-party package with a largely compatible API and additional options.

| Capability | `json` | `simplejson` |
|---|---|---|
| Installation | Standard library | `pip install simplejson` |
| Core API | `dump`, `dumps`, `load`, `loads` | Mostly compatible |
| `Decimal` | Requires conversion/default handler | Native with `use_decimal=True` |
| Object method | No automatic `for_json()` convention | Supports `for_json=True` |
| Non-finite numbers | `allow_nan` option | Also offers `ignore_nan` |
| Large JavaScript integers | No special conversion option | Offers `bigint_as_string` |

Do not add a dependency merely because it exists. Choose `simplejson` when one of its additional behaviours is an actual project requirement.

## Install `simplejson` when required

Run the following once in the notebook environment if the import cell reports that it is unavailable:

```python
%pip install simplejson
```

Restart the kernel after installation if the environment requests it.

In [ ]:
try:
    import simplejson
except ImportError:
    simplejson = None
    print("simplejson is not installed; run: %pip install simplejson")
else:
    print("simplejson version:", simplejson.__version__)

## `simplejson` with `Decimal`

With `use_decimal=True`, `simplejson` writes a `Decimal` as a JSON number without first converting it to `float`. When loading, `use_decimal=True` parses JSON floating-point numbers as `Decimal`.

In [ ]:
if simplejson is not None:
    decimal_payload = {"order_id": 5002, "total": Decimal("1250.75")}
    decimal_json = simplejson.dumps(
        decimal_payload,
        use_decimal=True,
        indent=2,
    )
    restored_payload = simplejson.loads(decimal_json, use_decimal=True)

    print(decimal_json)
    print(type(restored_payload["total"]))
else:
    print("Skipped: install simplejson to run this example.")

## `simplejson` and `for_json()`

When `for_json=True`, `simplejson` looks for a `for_json()` method on an unsupported object and encodes the value returned by that method.

This is a `simplejson` convention, not a method recognized by the standard-library `json` module. `to_dict()` remains an explicit, library-independent application convention.

In [ ]:
class Customer:
    def __init__(self, customer_id, name, joined_at):
        self.customer_id = int(customer_id)
        self.name = name
        self.joined_at = joined_at

    def for_json(self):
        return {
            "customer_id": self.customer_id,
            "name": self.name,
            "joined_at": self.joined_at.isoformat(),
        }


new_customer = Customer(
    customer_id=103,
    name="München Stores",
    joined_at=datetime(2026, 7, 28, tzinfo=timezone.utc),
)

if simplejson is not None:
    print(
        simplejson.dumps(
            new_customer,
            for_json=True,
            ensure_ascii=False,
            indent=2,
        )
    )
else:
    print("Skipped: install simplejson to run this example.")

# JSON conventions for application data

- Use UTF-8 for files and network payloads.
- Use `ensure_ascii=False` when human-readable Unicode output is wanted.
- Use consistent field names; `snake_case` is natural for Python systems.
- Store dates and times as ISO 8601 strings with an explicit timezone.
- Decide deliberately whether precise decimal values are strings or JSON numbers.
- Do not serialize secrets, passwords, tokens, or internal attributes.
- Treat decoded JSON as untrusted input: validate required fields, types, ranges, and sizes.
- Add a schema/version field when a persisted format is expected to evolve.
- JSON object keys are strings. Integer dictionary keys return as strings after a round trip.
- JSON cannot preserve tuples, sets, class types, or datetimes without an explicit conversion contract.
- Use JSON Lines for append-oriented records; use a JSON array/object for one complete document.

## Final recap

```python
json.dumps(value)                 # Python value -> JSON string
json.loads(text)                  # JSON string -> Python value
json.dump(value, file)            # Python value -> JSON file
json.load(file)                   # JSON file -> Python value
json.dumps(obj, default=handler)  # custom encoding
json.loads(text, object_hook=hook)# custom reconstruction
```

For class objects, an explicit `to_dict()` / `from_dict()` contract is the clearest starting point. Add `default`, `object_hook`, or a custom encoder only when automatic conversion improves the design.

## References

- [Python `json` documentation](https://docs.python.org/3/library/json.html)
- [`simplejson` documentation](https://simplejson.readthedocs.io/)